In [1]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import SDWriter, SDMolSupplier

In [6]:
# ── 1. Load CSV ───────────────────────────────────────────────────────────────
df = pd.read_csv("stage2_1.csv")
print(f"Original: {len(df)} molecules")



Original: 5200 molecules


In [7]:
# ── 2. Keep only valid SMILES_state and parseable SMILES ─────────────────────
df = df[df["SMILES_state"] == 1].copy()

def is_valid(smi):
    mol = Chem.MolFromSmiles(str(smi))
    return mol is not None

df = df[df["SMILES"].apply(is_valid)].copy()
print(f"After validity filter: {len(df)} molecules")

After validity filter: 2889 molecules


In [8]:
# ── 3. Canonicalize and deduplicate ───────────────────────────────────────────
df["canonical_smiles"] = df["SMILES"].apply(
    lambda s: Chem.MolToSmiles(Chem.MolFromSmiles(s), canonical=True)
)
df = df.drop_duplicates(subset="canonical_smiles").copy()
print(f"After deduplication: {len(df)} molecules")


After deduplication: 2381 molecules


In [9]:
# ── 4. Per-score threshold filtering ─────────────────────────────────────────

df = df[df["AutoDockGPU (raw)"] <= -3]

# Normalized component scores: must be > 0 (molecule passed each filter)
# Raise these thresholds (e.g. 0.5) to be more selective
normalized_score_cols = [
    "# rotatable bonds (<10)",
    "# stereo centers (<2)",
    "Molecular weight (350-600)",
    "logP (<=5)",
    "TPSA (<=150)",
    "# aromatic rings (<5)",
    "# Cl (<=1)",
    "# F (<4)",
    "# HB-donors (<5)",
    "# HB-acceptors (<10)",
    "AutoDockGPU",
    "Minimal Alerts",
]
threshold = 0.0  # adjust upward (e.g. 0.5) to be more stringent
for col in normalized_score_cols:
    df = df[df[col] > threshold]

print(f"After score thresholds: {len(df)} molecules")


After score thresholds: 15 molecules


In [10]:
# ── 5. Save filtered CSV ──────────────────────────────────────────────────────
df.to_csv("filtered_molecules.csv", index=False)
print("Saved: filtered_molecules.csv")


Saved: filtered_molecules.csv


In [11]:
import pandas as pd
from pathlib import Path
from rdkit import Chem
from rdkit.Chem import SDMolSupplier, SDWriter

df = pd.read_csv("filtered_molecules.csv")

def norm_smiles(s):
    mol = Chem.MolFromSmiles(str(s))
    if mol is None:
        return None
    return Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)

df["norm_smiles"] = df["SMILES"].apply(norm_smiles)

# lookup CSV rows by normalized SMILES
csv_lookup = {}
for i, row in df.iterrows():
    if pd.notna(row["norm_smiles"]):
        csv_lookup.setdefault(row["norm_smiles"], []).append(i)

seen = set()
records = []

for sdf_file in sorted(Path("poses").glob("best_pose-*.sdf")):
    supplier = SDMolSupplier(str(sdf_file), removeHs=False)

    for pose_index, mol in enumerate(supplier, start=1):
        if mol is None:
            continue

        if mol.HasProp("smiles"):
            smi = mol.GetProp("smiles")
        elif mol.HasProp("SMILES"):
            smi = mol.GetProp("SMILES")
        else:
            smi = Chem.MolToSmiles(Chem.RemoveHs(mol), canonical=True, isomericSmiles=True)

        smi_norm = norm_smiles(smi)
        if smi_norm is None:
            continue

        # keep only first match for each unique molecule
        if smi_norm in seen:
            continue

        if smi_norm in csv_lookup:
            seen.add(smi_norm)
            for idx in csv_lookup[smi_norm]:
                rec = df.loc[idx].to_dict()
                rec["source_sdf"] = sdf_file.name
                rec["pose_index"] = pose_index
                rec["sdf_smiles"] = smi
                records.append(rec)
            # only keep one pose per molecule, so stop after first match
            continue

matched_unique = pd.DataFrame(records)
matched_unique = matched_unique.drop_duplicates(subset="norm_smiles")
matched_unique.to_csv("filtered_matched_unique.csv", index=False)

print("Unique matched molecules:", len(matched_unique))
print("Saved: filtered_matched_unique.csv")

Unique matched molecules: 11
Saved: filtered_matched_unique.csv
